# Exercício 9 — QuickSort e QuickSelect com contagem de operações e variante em lista encadeada

Aluno: Henrique Goldstein Maluhy Mendes Amaral · DR1_AT · Infnet

Para rodar: Ambiente de execução → Executar tudo.

In [1]:
import math
import random
import time

import matplotlib.pyplot as plt
import pandas as pd

random.seed(42)


class Contador:
    def __init__(self):
        self.comparacoes = 0
        self.copias = 0
        self.chamadas = 0

## 1. O QuickSort

- A partição separa o trecho em três partes: menores, iguais e maiores que o pivô. É a mesma ideia do quicksort da aula, só que feita dentro do próprio vetor. Os iguais já ficam no lugar certo e não entram na recursão.
- Pivô: a mediana de três elementos sorteados. Primeiro usei o primeiro, o do meio e o último, mas a medição mostrou que no vetor ordenado o custo crescia mais que n log n: a partição deixava os trechos ordenados com o menor elemento no fim, e aí essa mediana escolhia sempre um pivô ruim. Sorteando as posições, nenhuma ordem de entrada engana a escolha.
- Trechos com até `short` elementos vão para o insertion sort.
- A recursão vai só para a parte menor e a maior é tratada num laço, então a pilha de chamadas fica em O(log n).

Comparação é comparação de chaves; cópia é escrita no vetor (uma troca = 3 cópias).

In [2]:
def trocar(lista, i, j, cont):
    if i != j:                                # trocar um elemento com ele mesmo não move nada
        lista[i], lista[j] = lista[j], lista[i]
        cont.copias += 3


def insertion_sort_trecho(lista, inicio, fim, cont):
    for i in range(inicio + 1, fim + 1):
        chave = lista[i]
        cont.copias += 1
        j = i - 1
        while j >= inicio:
            cont.comparacoes += 1
            if lista[j] <= chave:
                break
            lista[j + 1] = lista[j]
            cont.copias += 1
            j -= 1
        lista[j + 1] = chave
        cont.copias += 1


def escolher_pivo(lista, inicio, fim, estrategia, cont):
    """Devolve o VALOR do pivô."""
    if estrategia == "ultimo":
        return lista[fim]
    # três posições sorteadas; o pivô é o valor do meio entre os três
    a, b, c = (lista[random.randint(inicio, fim)] for _ in range(3))
    cont.comparacoes += 3                     # conto as 3 comparações da mediana
    if a <= b <= c or c <= b <= a:
        return b
    if b <= a <= c or c <= a <= b:
        return a
    return c


def particionar(lista, inicio, fim, estrategia, cont):
    """Três vias. Devolve (lt, gt): o trecho lista[lt..gt] é todo igual ao pivô."""
    pivo = escolher_pivo(lista, inicio, fim, estrategia, cont)
    # lt: fim da região dos menores · i: próximo a examinar · gt: começo da região dos maiores
    lt, i, gt = inicio, inicio, fim
    while i <= gt:
        cont.comparacoes += 1
        if lista[i] < pivo:
            trocar(lista, lt, i, cont)         # vai para a região dos menores
            lt += 1
            i += 1
        else:
            cont.comparacoes += 1
            if lista[i] > pivo:
                trocar(lista, i, gt, cont)     # vai para a região dos maiores
                gt -= 1
            else:
                i += 1                         # igual ao pivô: fica no meio
    return lt, gt


def _quicksort(lista, inicio, fim, short, estrategia, cont):
    cont.chamadas += 1
    while inicio < fim:
        if fim - inicio + 1 <= short:
            insertion_sort_trecho(lista, inicio, fim, cont)
            return
        lt, gt = particionar(lista, inicio, fim, estrategia, cont)
        # recursão no lado menor, laço no lado maior: a pilha fica em O(log n)
        if lt - inicio < fim - gt:
            _quicksort(lista, inicio, lt - 1, short, estrategia, cont)
            inicio = gt + 1
        else:
            _quicksort(lista, gt + 1, fim, short, estrategia, cont)
            fim = lt - 1


def quicksort(arr, short, pivo="mediana3"):
    """Ordena uma cópia de arr. Devolve (lista_ordenada, comparacoes, copias, chamadas)."""
    if short < 0:
        raise ValueError(f"short não pode ser negativo: {short}")
    lista = arr.copy()
    cont = Contador()
    _quicksort(lista, 0, len(lista) - 1, short, pivo, cont)
    return lista, cont.comparacoes, cont.copias, cont.chamadas


for _ in range(300):
    arr = [random.randint(0, 30) for _ in range(random.randint(0, 150))]
    for short in (0, 8, 32):
        assert quicksort(arr, short)[0] == sorted(arr)
print("quicksort conferido contra sorted() em 300 listas sorteadas, com repetidos, para short = 0, 8 e 32")

quicksort conferido contra sorted() em 300 listas sorteadas, com repetidos, para short = 0, 8 e 32


## 2. Quatro padrões × três valores de `short`

In [3]:
def gerar(n, padrao):
    if padrao == "ordenado":
        return list(range(n))
    if padrao == "reverso":
        return list(range(n - 1, -1, -1))
    if padrao == "aleatorio":
        lista = list(range(n))
        random.shuffle(lista)
        return lista
    if padrao == "quase ordenado":
        lista = list(range(n))
        for _ in range(n // 20):
            i = random.randrange(n)
            j = min(n - 1, i + random.randint(1, 10))
            lista[i], lista[j] = lista[j], lista[i]
        return lista
    raise ValueError(padrao)


PADROES = ["ordenado", "reverso", "quase ordenado", "aleatorio"]
linhas = []
for n in [1_000, 4_000, 16_000]:
    for padrao in PADROES:
        entrada = gerar(n, padrao)
        for short in (0, 8, 32):
            inicio = time.perf_counter()
            _, comparacoes, copias, chamadas = quicksort(entrada, short)
            tempo = time.perf_counter() - inicio
            linhas.append({
                "n": n, "padrao": padrao, "short": short,
                "comparacoes": comparacoes, "copias": copias, "chamadas": chamadas,
                "comparacoes / (n·log2 n)": comparacoes / (n * math.log2(n)),
                "tempo_ms": tempo * 1000,
            })

qs = pd.DataFrame(linhas)
qs[qs["n"] == 16_000]

,n,padrao,short,comparacoes,copias,chamadas,comparacoes / (n·log2 n),tempo_ms
24,16000,ordenado,0,384160,348042,10093,1.719202,45.9473
25,16000,ordenado,8,363183,416458,2834,1.625325,33.5149
26,16000,ordenado,32,350245,373192,822,1.567425,31.5785
27,16000,reverso,0,392561,384144,10068,1.756798,47.6534
28,16000,reverso,8,349472,391650,2814,1.563965,40.0930
29,16000,reverso,32,350752,396250,812,1.569693,29.3445
30,16000,quase ordenado,0,390545,381870,10087,1.747776,47.3351
31,16000,quase ordenado,8,340139,348610,2831,1.522198,32.3491
32,16000,quase ordenado,32,368427,392508,823,1.648793,31.2045
33,16000,aleatorio,0,387594,493293,10157,1.734570,48.0966


Com `short` maior as chamadas recursivas caem muito (de cerca de 10 mil para 800, com n = 16.000), as comparações caem um pouco e o tempo também. É ganho de constante: o custo continua O(n log n).

Comparações / (n·log₂ n) por padrão de entrada:

In [4]:
resumo = qs[qs["short"] == 0].pivot(index="n", columns="padrao", values="comparacoes / (n·log2 n)")
print("comparações / (n·log2 n), com short = 0:\n")
print(resumo.round(3))

comparações / (n·log2 n), com short = 0:

padrao  aleatorio  ordenado  quase ordenado  reverso
n                                                   
1000        1.711     1.750           1.645    1.687
4000        1.724     1.755           1.709    1.779
16000       1.735     1.719           1.748    1.757


Os quatro padrões ficam na mesma faixa: com o pivô sorteado, a ordem da entrada não muda o custo.

## 3. O pior caso: pivô ruim

Com o pivô sempre no último elemento, num vetor já ordenado cada partição separa um elemento só:

In [5]:
linhas = []
for n in [500, 1000, 2000]:
    entrada = gerar(n, "ordenado")
    _, c_ingenuo, _, _ = quicksort(entrada, 0, pivo="ultimo")
    _, c_mediana, _, _ = quicksort(entrada, 0, pivo="mediana3")
    linhas.append({
        "n": n,
        "pivo_ultimo": c_ingenuo, "pivo_ultimo / n²": c_ingenuo / n**2,
        "mediana_de_tres": c_mediana, "mediana / (n·log2 n)": c_mediana / (n * math.log2(n)),
    })
pd.DataFrame(linhas)

,n,pivo_ultimo,pivo_ultimo / n²,mediana_de_tres,mediana / (n·log2 n)
0,500,125748,0.502992,7569,1.688419
1,1000,501498,0.501498,17709,1.776980
2,2000,2002998,0.500749,38851,1.771465


Com o pivô no último, `comparacoes / n²` fica em 0,5: O(n²). Com a mediana de três sorteados, a mesma entrada fica em O(n log n). E mesmo no pior caso a pilha não estoura, porque a recursão vai só para a parte menor.

## 4. QuickSelect

Usa a mesma partição, mas depois de cada partição segue só para o lado onde está a posição procurada.

In [6]:
def quickselect(arr, goal_index):
    """Devolve (elemento, comparacoes, copias)."""
    if not 0 <= goal_index < len(arr):
        raise IndexError(f"goal_index {goal_index} fora do vetor de tamanho {len(arr)}")
    lista = arr.copy()
    cont = Contador()
    inicio, fim = 0, len(lista) - 1
    while inicio < fim:
        lt, gt = particionar(lista, inicio, fim, "mediana3", cont)
        if goal_index < lt:
            fim = lt - 1               # está entre os menores
        elif goal_index > gt:
            inicio = gt + 1            # está entre os maiores
        else:
            break                      # caiu no bloco dos iguais ao pivô: achou
    return lista[goal_index], cont.comparacoes, cont.copias


# confere contra a posição no vetor ordenado, com repetidos
for _ in range(300):
    arr = [random.randint(0, 40) for _ in range(random.randint(1, 120))]
    k = random.randrange(len(arr))
    assert quickselect(arr, k)[0] == sorted(arr)[k]
print("quickselect conferido em 300 casos sorteados")

linhas = []
for n in [1_000, 10_000, 100_000]:
    entrada = gerar(n, "aleatorio")
    _, c_sort, cp_sort, _ = quicksort(entrada, 0)
    for nome, k in [("mínimo", 0), ("mediana", n // 2), ("percentil 90", int(n * 0.9))]:
        elemento, c_sel, cp_sel = quickselect(entrada, k)
        assert elemento == k                   # os valores são 0..n-1
        linhas.append({
            "n": n, "posicao": nome,
            "quickselect_comparacoes": c_sel, "quickselect / n": c_sel / n,
            "ordenar_tudo_comparacoes": c_sort,
            "ganho (ordenar / selecionar)": c_sort / c_sel,
            "quickselect_copias": cp_sel, "ordenar_tudo_copias": cp_sort,
        })
pd.DataFrame(linhas)

quickselect conferido em 300 casos sorteados


,n,posicao,quickselect_comparacoes,quickselect / n,ordenar_tudo_comparacoes,ganho (ordenar / selecionar),quickselect_copias,ordenar_tudo_copias
0,1000,mínimo,3667,3.66700,17497,4.771475,5712,21882
1,1000,mediana,3223,3.22300,17497,5.428793,4446,21882
2,1000,percentil 90,4121,4.12100,17497,4.245814,5817,21882
3,10000,mínimo,30937,3.09370,232409,7.512332,46629,294417
4,10000,mediana,44138,4.41380,232409,5.265508,66267,294417
5,10000,percentil 90,24462,2.44620,232409,9.500818,36969,294417
6,100000,mínimo,317500,3.17500,2952180,9.298205,508827,3806982
7,100000,mediana,341375,3.41375,2952180,8.647909,362157,3806982
8,100000,percentil 90,220342,2.20342,2952180,13.398172,383334,3806982


O QuickSelect fica em poucas comparações por elemento, sem crescer com n: O(n) em média (n + n/2 + n/4 + ... ≈ 2n). Ordenar tudo é O(n log n), então o ganho cresce com n. O pior caso do QuickSelect também é O(n²), improvável com o pivô sorteado.

## 5. Variante em `SinglyLinkedList`

Sem acesso por índice, a partição desliga cada nó e liga no fim de uma de três listas (menores, iguais e maiores), como no quicksort da aula. Cada nó entra no fim da sua lista na ordem em que foi encontrado, então a partição é estável.

In [7]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None


class SinglyLinkedList:
    def __init__(self, valores=()):
        self.head = None
        self.tail = None
        for v in valores:
            self.append_node(Node(v))

    def append_node(self, no):
        """Anexa um nó existente no fim. O(1) graças ao ponteiro tail."""
        no.next = None
        if self.head is None:
            self.head = no
        else:
            self.tail.next = no
        self.tail = no

    def to_list(self):
        valores, atual = [], self.head
        while atual is not None:
            valores.append(atual.value)
            atual = atual.next
        return valores


def particionar_lista(lista, pivo, chave, cont):
    """Separa os nós em três listas novas, mantendo a ordem relativa (estável)."""
    menores, iguais, maiores = SinglyLinkedList(), SinglyLinkedList(), SinglyLinkedList()
    atual = lista.head
    while atual is not None:
        proximo = atual.next                  # guardo antes de religar o nó
        cont.comparacoes += 1
        if chave(atual.value) < pivo:
            menores.append_node(atual)
        else:
            cont.comparacoes += 1
            if chave(atual.value) > pivo:
                maiores.append_node(atual)
            else:
                iguais.append_node(atual)
        atual = proximo                       # nenhum valor copiado: o nó só mudou de lista
    return menores, iguais, maiores


def concatenar(*listas):
    resultado = SinglyLinkedList()
    for l in listas:
        if l.head is None:
            continue
        if resultado.head is None:
            resultado.head = l.head
        else:
            resultado.tail.next = l.head      # junta em O(1), sem percorrer nada
        resultado.tail = l.tail
    return resultado


def quicksort_lista(lista, cont, chave=lambda v: v):
    cont.chamadas += 1
    if lista.head is None or lista.head.next is None:
        return lista
    pivo = chave(lista.head.value)            # o primeiro nó: é o único de acesso O(1)
    menores, iguais, maiores = particionar_lista(lista, pivo, chave, cont)
    return concatenar(quicksort_lista(menores, cont, chave), iguais, quicksort_lista(maiores, cont, chave))


# estabilidade: pares (chave, ordem de chegada); os de mesma chave precisam manter a ordem
pares = [(3, "a"), (1, "b"), (3, "c"), (2, "d"), (1, "e"), (3, "f")]
cont = Contador()
m, i, M = particionar_lista(SinglyLinkedList(pares), 2, lambda p: p[0], cont)
print("pivô 2 →  menores:", m.to_list())
print("          iguais: ", i.to_list())
print("          maiores:", M.to_list())
ordenado = quicksort_lista(SinglyLinkedList(pares), Contador(), chave=lambda p: p[0]).to_list()
print("\nordenado pela chave:", ordenado)
assert ordenado == sorted(pares, key=lambda p: p[0])   # o sorted do Python é estável: mesmo resultado

for _ in range(200):
    arr = [random.randint(0, 20) for _ in range(random.randint(0, 100))]
    assert quicksort_lista(SinglyLinkedList(arr), Contador()).to_list() == sorted(arr)
print("quicksort em lista conferido em 200 listas sorteadas")

pivô 2 →  menores: [(1, 'b'), (1, 'e')]
          iguais:  [(2, 'd')]
          maiores: [(3, 'a'), (3, 'c'), (3, 'f')]

ordenado pela chave: [(1, 'b'), (1, 'e'), (2, 'd'), (3, 'a'), (3, 'c'), (3, 'f')]
quicksort em lista conferido em 200 listas sorteadas


### Por que o custo muda sem acesso por índice

- A partição continua O(n), mas religando nós em vez de trocar valores: nenhum valor é copiado.
- Escolher um bom pivô (o do meio, uma mediana ou um sorteado) custa O(1) no vetor e O(n) na lista, porque é preciso andar até a posição. Por isso na lista o pivô natural é o primeiro nó.
- Com o primeiro nó como pivô, uma lista já ordenada cai no pior caso, O(n²):

In [8]:
linhas = []
for n in [150, 300, 600]:       # mais do que isso, a recursão na lista ordenada passa do limite do Python
    for padrao in ["aleatorio", "ordenado"]:
        cont = Contador()
        resultado = quicksort_lista(SinglyLinkedList(gerar(n, padrao)), cont)
        assert resultado.to_list() == list(range(n))
        linhas.append({
            "n": n, "padrao": padrao, "comparacoes": cont.comparacoes,
            "comparacoes / (n·log2 n)": cont.comparacoes / (n * math.log2(n)),
            "comparacoes / n²": cont.comparacoes / n**2,
        })
pd.DataFrame(linhas)

,n,padrao,comparacoes,comparacoes / (n·log2 n),comparacoes / n²
0,150,aleatorio,1795,1.655411,0.079778
1,150,ordenado,22648,20.886769,1.006578
2,300,aleatorio,3915,1.585890,0.043500
3,300,ordenado,90298,36.577952,1.003311
4,600,aleatorio,9611,1.735686,0.026697
5,600,ordenado,360598,65.121733,1.001661


Na lista aleatória o custo é O(n log n), como no vetor. Na lista já ordenada é O(n²) (`comparacoes / n²` estável). No vetor a mediana evitou isso porque custa O(1); na lista, a mesma proteção custaria O(n) por partição. A recursão também chega a profundidade n na lista ordenada, por isso parei em n = 600.

## 6. Casos de borda

In [9]:
for short in (0, 8, 32):
    assert quicksort([], short)[0] == []
    assert quicksort([5], short)[0] == [5]
    assert quicksort([2, 2, 2, 2], short)[0] == [2, 2, 2, 2]
    assert quicksort([3, -1, 0, -1], short)[0] == [-1, -1, 0, 3]

# vetor todo igual: a partição de três vias resolve numa passada só
_, comparacoes, _, chamadas = quicksort([7] * 10_000, 0)
print(f"10.000 elementos iguais: {comparacoes} comparações, {chamadas} chamada(s) — linear")

assert quickselect([9], 0)[0] == 9
assert quickselect([4, 4, 4], 1)[0] == 4
for invalido in (-1, 3):
    try:
        quickselect([1, 2, 3], invalido)
    except IndexError as erro:
        print(f"quickselect(goal_index={invalido}) → {erro}")
try:
    quicksort([1, 2], -1)
except ValueError as erro:
    print(f"quicksort(short=-1) → {erro}")

assert quicksort_lista(SinglyLinkedList(), Contador()).to_list() == []
assert quicksort_lista(SinglyLinkedList([1]), Contador()).to_list() == [1]
print("\ntodos os casos de borda passaram")

10.000 elementos iguais: 20003 comparações, 2 chamada(s) — linear
quickselect(goal_index=-1) → goal_index -1 fora do vetor de tamanho 3
quickselect(goal_index=3) → goal_index 3 fora do vetor de tamanho 3
quicksort(short=-1) → short não pode ser negativo: -1

todos os casos de borda passaram
